# Demand Forecasting Orchestator

Model orchestrator for mensual runnings.

## Data Model Pipeline

In [1]:
%%bash
uv sync

Resolved 71 packages in 1ms
Installed 2 packages in 10ms
 + nvidia-nccl-cu12==2.30.4
 + xgboost==3.2.0


In [2]:
%%bash
set -e

BUCKET="itam-anlytics-karla"

echo "==== BRONZE ===="
uv run python elt/bronze.py \
  --bucket $BUCKET \
  --secret-name kaggle/credentials \
  --region-name us-east-1

echo "==== SILVER ===="
uv run python elt/silver.py \
  --bucket $BUCKET

echo "==== GOLD ===="
uv run python elt/gold.py \
  --bucket $BUCKET

==== BRONZE ====


2026-05-04 02:50:07,363 - INFO - Starting raw data extraction
2026-05-04 02:50:07,363 - INFO - Loading Kaggle credentials
2026-05-04 02:50:07,501 - INFO - Downloading Kaggle data
2026-05-04 02:50:07,784 - INFO - Kaggle files downloaded to: /home/sagemaker-user/.cache/kagglehub/competitions/competitive-data-science-predict-future-sales
2026-05-04 02:50:07,784 - INFO - Loading raw data to s3
2026-05-04 02:50:07,784 - INFO - Loading raw data
2026-05-04 02:50:07,832 - INFO - Uploading item_categories.csv to s3://itam-anlytics-karla/sales_predict/bronze/item_categories.csv
2026-05-04 02:50:07,912 - INFO - Loaded successfully
2026-05-04 02:50:07,912 - INFO - Uploading items.csv to s3://itam-anlytics-karla/sales_predict/bronze/items.csv
2026-05-04 02:50:07,992 - INFO - Loaded successfully
2026-05-04 02:50:07,992 - INFO - Uploading sales_train.csv to s3://itam-anlytics-karla/sales_predict/bronze/sales_train.csv
2026-05-04 02:50:08,560 - INFO - Loaded successfully
2026-05-04 02:50:08,561 - INFO

==== SILVER ====


2026-05-04 02:50:09,937 - INFO - Starting Silver layer processing
2026-05-04 02:50:09,937 - INFO - Reading sales from s3://itam-anlytics-karla/sales_predict/bronze/sales_train.csv
2026-05-04 02:50:12,642 - INFO - Cleaning sales
2026-05-04 02:50:12,643 - INFO - Checking that all columns are present
2026-05-04 02:50:12,643 - INFO - Keeping only the necessary columns
2026-05-04 02:50:13,108 - INFO - Casting to the expected data type
2026-05-04 02:50:27,956 - INFO - Removing duplicates
2026-05-04 02:50:28,983 - WARNING - sales: removed 6 duplicated rows
2026-05-04 02:50:28,983 - INFO - Checking for empty columns
2026-05-04 02:50:28,992 - INFO - Validation of unique IDs
2026-05-04 02:50:28,992 - INFO - sales cleaned successfully: 2935843 rows
2026-05-04 02:50:28,992 - INFO - Writing sales to s3://itam-anlytics-karla/sales_predict/silver/sales/
2026-05-04 02:50:33,092 - INFO - Reading items from s3://itam-anlytics-karla/sales_predict/bronze/items.csv
2026-05-04 02:50:33,464 - INFO - Cleaning

==== GOLD ====


2026-05-04 02:50:38,061 - INFO - Starting Gold layer processing
2026-05-04 02:50:38,061 - INFO - Reading sales from s3://itam-anlytics-karla/sales_predict/silver/sales/
2026-05-04 02:50:39,200 - INFO - Reading items from s3://itam-anlytics-karla/sales_predict/silver/items/
2026-05-04 02:50:39,424 - INFO - Reading item_categories from s3://itam-anlytics-karla/sales_predict/silver/item_categories/
2026-05-04 02:50:39,664 - INFO - Reading shops from s3://itam-anlytics-karla/sales_predict/silver/shops/
2026-05-04 02:50:39,885 - INFO - Creating monthly sales table
2026-05-04 02:50:40,581 - INFO - Monthly sales created: 1609124 rows
2026-05-04 02:50:40,600 - INFO - Merging monthly sales with catalogs
2026-05-04 02:50:41,272 - INFO - Gold table after merges: 1609124 rows
2026-05-04 02:50:41,272 - INFO - Adding naive baseline features
2026-05-04 02:50:43,430 - INFO - Naive baseline features added
2026-05-04 02:50:43,449 - INFO - Identifying inactive items
2026-05-04 02:50:45,460 - INFO - Inact

## Machine Learning Pipeline

In [3]:
%%bash
set -e

BUCKET="itam-anlytics-karla"

echo "==== TRAIN ===="
uv run python models/xgboost_train.py \
  --bucket $BUCKET

echo "==== PREDICT ===="
uv run python models/xgboost_predict.py \
  --bucket $BUCKET

==== TRAIN ====


2026-05-04 02:50:56,115 - INFO - Starting XGBoost training
2026-05-04 02:50:56,115 - INFO - Reading Gold data from s3://itam-anlytics-karla/sales_predict/gold/modeling_sales/
2026-05-04 02:50:57,629 - INFO - Preparing training data
2026-05-04 02:50:57,790 - INFO - Rows before dropna: 778911
2026-05-04 02:50:57,800 - INFO - Rows after dropna: 778911
2026-05-04 02:50:57,803 - INFO - Training data prepared: 778911 rows
2026-05-04 02:50:57,804 - INFO - Training XGBoost model
2026-05-04 02:50:59,946 - INFO - Model training completed
2026-05-04 02:50:59,947 - INFO - Evaluating backtesting performance
2026-05-04 02:50:59,959 - INFO - Backtesting metrics:
    model_name  backtest_month  mae_model  rmse_model  mae_naive  rmse_naive  mae_improvement  rmse_improvement
xgboost_simple              33   1.074986     2.11651   1.437959    2.909305         0.362972          0.792795
2026-05-04 02:50:59,959 - INFO - Evaluating backtesting by ['item_category_id']
2026-05-04 02:51:00,103 - INFO - Evaluat

==== PREDICT ====


2026-05-04 02:51:02,827 - INFO - Starting XGBoost inference
2026-05-04 02:51:02,827 - INFO - Reading Silver test data from s3://itam-anlytics-karla/sales_predict/silver/test/
2026-05-04 02:51:03,181 - INFO - Reading Gold data from s3://itam-anlytics-karla/sales_predict/gold/modeling_sales/
2026-05-04 02:51:04,525 - INFO - Loading model from s3://itam-anlytics-karla/sales_predict/models/xgboost_model.joblib
2026-05-04 02:51:05,862 - INFO - Creating forecast dataset for month 34
2026-05-04 02:51:07,584 - WARNING - Filling 102796 rows with missing features
2026-05-04 02:51:07,596 - INFO - Generating predictions
2026-05-04 02:51:07,782 - INFO - Writing predictions to s3://itam-anlytics-karla/sales_predict/predictions/xgboost_forecast/
2026-05-04 02:51:08,426 - INFO - XGBoost inference completed successfully
